In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from lightgbm import LGBMRegressor, log_evaluation
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split, KFold, cross_validate, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

from features.FeatureNameCleaner import FeatureNameCleaner
from features.preprocess import build_preprocessor
from features.feature_engineering import build_features, save_train_reference

from config.setting import MODEL_DIR, NUM_FEATURES
from utils.helper import get_data_path, get_project_root
import seaborn as sns

In [2]:
df = pd.read_csv(get_data_path("final_data.csv"))
print(f"Before cleaning: {len(df)} rows")

df = df.drop_duplicates()
df = df[(df.price > df.price.quantile(0.01)) &
        (df.price < df.price.quantile(0.99))]
df = df[(df["floors"] <= 15) | (df["floors"].isna())]

print(f"After cleaning: {len(df)} rows")

Before cleaning: 20356 rows
After cleaning: 19904 rows


In [3]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
save_train_reference(train_df)

train_df = build_features(train_df, mode="train")
test_df = build_features(test_df, mode="predict")

X_train = train_df.drop("price", axis=1)
y_train = np.log1p(train_df["price"] * 1000 / train_df["area"])

X_test = test_df.drop("price", axis=1)
y_test = np.log1p(test_df["price"] * 1000 / test_df["area"])
print(f"\nTrain size: {len(X_train)}, Test size: {len(X_test)}")
print(f"Features: {X_train.shape[1]} columns")


Train size: 15923, Test size: 3981
Features: 30 columns


In [4]:
model = LGBMRegressor(
    learning_rate=0.05,
    n_estimators=1500,
    num_leaves=31,
    max_depth=8,
    min_child_samples=30,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.5,
    random_state=42,
    verbose=-1,
)

pipe = Pipeline([
    ("prep", build_preprocessor()),
    ("clean_names", FeatureNameCleaner()),
    ("model", model)
])

In [5]:
from lightgbm import early_stopping

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.1, random_state=42
)

preprocessor = pipe.named_steps["prep"]
cleaner = pipe.named_steps["clean_names"]
model = pipe.named_steps["model"]

X_tr_processed = cleaner.fit_transform(preprocessor.fit_transform(X_tr))
X_val_processed = cleaner.transform(preprocessor.transform(X_val))

model.fit(
    X_tr_processed, y_tr,
    eval_set=[(X_val_processed, y_val)],
    callbacks=[
        early_stopping(stopping_rounds=50),
        log_evaluation(period=100)
    ]
)

print(f"\nBest iteration: {model.best_iteration_}")
print(f"Best score (val): {model.best_score_}")

Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 0.0378515
[200]	valid_0's l2: 0.0355158
[300]	valid_0's l2: 0.0348799
[400]	valid_0's l2: 0.0343709
[500]	valid_0's l2: 0.0342204
[600]	valid_0's l2: 0.0340862
[700]	valid_0's l2: 0.0339601
[800]	valid_0's l2: 0.0338551
Early stopping, best iteration is:
[770]	valid_0's l2: 0.0338331

Best iteration: 770
Best score (val): defaultdict(<class 'collections.OrderedDict'>, {'valid_0': OrderedDict({'l2': np.float64(0.03383308116756672)})})


In [6]:
model = LGBMRegressor(
    learning_rate=0.05,
    n_estimators=1500,
    num_leaves=31,
    max_depth=8,
    min_child_samples=30,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.5,
    random_state=42,
    verbose=-1,
)

pipe_final = Pipeline([
    ("prep", build_preprocessor()),
    ("clean_names", FeatureNameCleaner()),
    ("model", model)
])
pipe_final.fit(X_train, y_train)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['year', 'log_area',
                                                   'bedrooms', 'bathrooms',
                                                   'floors', 'lat', 'lng',
                                                   'dist_center',
                                                   'log_dist_center',
                                                   'zone_price_mean',
                                                   'knn_price_mean',
                                                   'knn_price_median',
                                                   'knn_price_std',
                                                   'neighbor_density',
                                                   'area_zone_price',
                                                   'area_x_bedrooms',
                                                   'flo...
                                                                   OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                                  unknown_value=-1))]),
                                                  ['legal_status',
                                                   'furniture_state',
                                                   'property_type',
                                                   'property_feature'])])),
                ('clean_names', FeatureNameCleaner()),
                ('model',
                 LGBMRegressor(colsample_bytree=0.8, learning_rate=0.05,
                               max_depth=8, min_child_samples=30,
                               n_estimators=1500, random_state=42,
                               reg_alpha=0.1, reg_lambda=0.5, subsample=0.8,
                               verbose=-1))])

In [7]:
model = LGBMRegressor(
    learning_rate=0.05,
    n_estimators=1500,
    num_leaves=31,
    max_depth=8,
    min_child_samples=30,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.5,
    random_state=42,
    verbose=-1,
)

pipe = Pipeline([
    ("prep", build_preprocessor()),
    ("clean_names", FeatureNameCleaner()),
    ("model", model)
])

In [8]:
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_validate(
    pipe, X_train, y_train,
    cv=kfold,
    scoring={
        "mae": "neg_mean_absolute_error",
        "rmse": "neg_root_mean_squared_error",
        "r2": "r2"
    }
)

print("\n=== CROSS-VALIDATION RESULTS ===")
print(f"CV MAE:  {-scores['test_mae'].mean():.5f} (+/- {scores['test_mae'].std():.5f})")
print(f"CV RMSE: {-scores['test_rmse'].mean():.5f} (+/- {scores['test_rmse'].std():.5f})")
print(f"CV R2:   {scores['test_r2'].mean():.5f} (+/- {scores['test_r2'].std():.5f})")


=== CROSS-VALIDATION RESULTS ===
CV MAE:  0.13540 (+/- 0.00187)
CV RMSE: 0.18521 (+/- 0.00406)
CV R2:   0.80830 (+/- 0.00855)


In [9]:
train_pred_log = pipe_final.predict(X_train)
test_pred_log = pipe_final.predict(X_test)

print("\n=== TRAIN PERFORMANCE ===")
print(f"RMSE: {root_mean_squared_error(y_train, train_pred_log):.5f}")
print(f"MAE:  {mean_absolute_error(y_train, train_pred_log):.5f}")
print(f"R2:   {r2_score(y_train, train_pred_log):.5f}")

print("\n=== TEST PERFORMANCE ===")
print(f"RMSE: {root_mean_squared_error(y_test, test_pred_log):.5f}")
print(f"MAE:  {mean_absolute_error(y_test, test_pred_log):.5f}")
print(f"R2:   {r2_score(y_test, test_pred_log):.5f}")

y_pred_price_m2 = np.expm1(test_pred_log)
y_test_price_m2 = np.expm1(y_test)
mae_price_m2 = mean_absolute_error(y_test_price_m2, y_pred_price_m2)
print(f"\nMAE (price per m2): {mae_price_m2:.2f} triệu/m²")

y_pred_total = y_pred_price_m2 * test_df["area"].values / 1000
y_test_total = test_df["price"].values
mae_total = mean_absolute_error(y_test_total, y_pred_total)
print(f"MAE (total price): {mae_total:.2f} tỷ")

mape = np.mean(np.abs((y_test_total - y_pred_total) / (y_test_total + 1e-8))) * 100
print(f"MAPE: {mape:.1f}%")
print(f"MAPE: {mape:.1f}%")


=== TRAIN PERFORMANCE ===
RMSE: 0.10543
MAE:  0.07792
R2:   0.93793

=== TEST PERFORMANCE ===
RMSE: 0.18324
MAE:  0.13604
R2:   0.81260

MAE (price per m2): 15.08 triệu/m²
MAE (total price): 0.87 tỷ
MAPE: 14.2%
MAPE: 14.2%


In [1]:
y_pred_price_m2 = np.expm1(test_pred_log)
y_pred_total = y_pred_price_m2 * test_df["area"].values / 1000
y_test_total = test_df["price"].values

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(y_test_total, y_pred_total, alpha=0.3, s=10)
max_val = max(y_test_total.max(), y_pred_total.max())
axes[0].plot([0, max_val], [0, max_val], "r--", linewidth=1)
axes[0].set_xlabel("Actual Price (tỷ)")
axes[0].set_ylabel("Predicted Price (tỷ)")
axes[0].set_title("Prediction vs Actual")

residuals = y_test_total - y_pred_total
axes[1].scatter(y_pred_total, residuals, alpha=0.3, s=10)
axes[1].axhline(y=0, color="r", linestyle="--")
axes[1].set_xlabel("Predicted Price (tỷ)")
axes[1].set_ylabel("Residual (tỷ)")
axes[1].set_title("Residual Plot")

preprocessor = pipe_final.named_steps["prep"]
model = pipe_final.named_steps["model"]
feature_names = preprocessor.get_feature_names_out()
importances = model.feature_importances_

feat_imp = pd.Series(importances, index=feature_names)
feat_imp = feat_imp.sort_values(ascending=False)
top_features = feat_imp.head(20)
top_features.sort_values().plot(kind="barh", ax=axes[2])
axes[2].set_title("Top 20 Feature Importance")

plt.tight_layout()
print("\n=== TOP 20 FEATURES ===")
print(feat_imp.head(20).to_string())

NameError: name 'np' is not defined

In [12]:
model_path = Path(get_project_root()) / MODEL_DIR / "lgbm.pkl"
joblib.dump(pipe_final, model_path)
print(f"\nModel saved to: {model_path}")


Model saved to: D:\code\DA\TLand-Backend\rag-chatbot-service\models\lgbm.pkl
